# Multi-Tool Utility Agent

A small agent built from scratch to practice the core agent loop and tool-calling
concepts from Day 4 (AI Engineer Agentic Track).

**Tools:**
- `get_weather(city)` — fetches current weather via Open-Meteo API
- `calculate(expression)` — safely evaluates math expressions using `ast`
- `search_notes(query)` — keyword search over a local notes file

**Goal:** Watch the agent loop decide which tool(s) to call, chain multiple tool
calls in sequence, and feed results back into context before producing a final
answer. Each loop iteration is logged to show the full reasoning trace.

**Example prompts to test chaining:**
- "What's the weather in Tokyo, and what's 15% of the temperature in Celsius?"
- "Search my notes for anything about vacations, then tell me if I should pack
  a coat for wherever it mentions."

In [18]:
# Check the key - if you're not using OpenAI, check whichever key you're using! Ollama doesn't need a key.

import os
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")
    


OpenAI API Key exists and begins 5WPhu17L


In [ ]:
import ast
import json
import requests
from openai import OpenAI  # or whatever client the course uses

client = OpenAI()
# model is gpt-5.4-nano

In [6]:
def get_weather(city: str) -> str:
    # Open-Meteo needs lat/lon coordinates, so we need to convert the city name to coordinates first.
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}).json()
    if not geo.get("results"):
        return f"Could not find coordinates for city: {city}"
    lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]
    weather = requests.get("https://api.open-meteo.com/v1/forecast", params={"latitude": lat, "longitude": lon, "current_weather": True}).json()
    if not weather.get("current_weather"):
        return f"Could not retrieve weather data for city: {city}"
    temp = weather["current_weather"]["temperature"]
    return f"The current temperature in {city} is {temp}°C."

In [10]:
get_weather("Chennai")

'The current temperature in Chennai is 30.8°C.'

In [11]:
def calculate(expression: str) -> str:
    try:
        node = ast.parse(expression, mode='eval')
        # very basic safety: only allow numbers and operators
        result = eval(compile(node, '<string>', 'eval'))
        return str(result)
    except Exception as e:
        return f"Error occurred while evaluating expression: {e}"

In [13]:
calculate("2 + 2 * 3")

'8'

In [14]:
NOTES = """
Trip to Lisbon planned for October, remember to check the weather.
Need to buy a new suitcase before the vacation.
Reminder: pack layers for unpredictable autumn weather.
"""

In [27]:
def search_notes(query: str) -> str:
    query_words = query.lower().split()
    matches = [
        line.strip() for line in NOTES.splitlines()
        if line.strip() and any(word in line.lower() for word in query_words)
    ]
    return "\n".join(matches) if matches else "No matching notes found."

In [28]:
tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The name of the city to get the weather for."}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The mathematical expression to evaluate."}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_notes",
            "description": "Search through personal notes for a given query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query to look for in the notes."}
                },
                "required": ["query"]
            }
        }
    }
]

In [29]:
available_tools = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_notes": search_notes
}

In [30]:
def dispatch(name: str, args: dict) -> str:
    print(f"🔧 Tool called: {name}({args})")
    
    fn = available_tools.get(name)
    if not fn:
        print(f"   ⚠️ Unknown tool: {name}")
        return f"Unknown tool: {name}"
    
    result = fn(**args)
    print(f"   → Result: {result}")
    return result

In [31]:
def run_agent(user_message: str):
    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to tools."},
        {"role": "user", "content": user_message},
    ]

    while True:
        response = client.chat.completions.create(
            model="gpt-5.4-nano",  # swap for whatever model the course uses
            messages=messages,
            tools=tool_schemas,
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)  # record the assistant's tool call request
            for call in msg.tool_calls:
                args = json.loads(call.function.arguments)
                result = dispatch(call.function.name, args)

                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": result,
                })
            # loop continues so the model can see tool results
        else:
            print("\n💬 Final answer:", msg.content)
            break

In [23]:
run_agent("What's the weather in Tokyo, and what's 15% of the temperature in Celsius?")

🔧 Tool called: get_weather({'city': 'Tokyo'})
   → Result: The current temperature in Tokyo is 25.7°C.
🔧 Tool called: calculate({'expression': '0.15*1'})
   → Result: 0.15

💬 Final answer: - **Weather in Tokyo:** **25.7°C** right now.  
- **15% of a temperature in Celsius:** it’s **0.15 × (that Celsius temperature)**. (For example, if it’s *T*°C, then 15% is *0.15T*.)


In [34]:
run_agent("What's the weather in the city I visit in October (check my notes)? What's 15% of the temperature in Celsius?")

🔧 Tool called: search_notes({'query': 'city I visit in October'})
   → Result: Trip to Lisbon planned for October, remember to check the weather.
Need to buy a new suitcase before the vacation.
Reminder: pack layers for unpredictable autumn weather.
🔧 Tool called: get_weather({'city': 'Lisbon'})
   → Result: The current temperature in Lisbon is 28.8°C.
🔧 Tool called: calculate({'expression': '0.15*28.8'})
   → Result: 4.32

💬 Final answer: - **Your October city (from your notes): Lisbon.** The **current weather in Lisbon is 28.8°C**.  
- **15% of 28.8°C** = **4.32°C**.
